In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
def setup_driver():
  """Setup Chrome driver with options"""
  chrome_options = Options()
  chrome_options.page_load_strategy = "eager"

  service = Service(ChromeDriverManager().install())
  driver = webdriver.Chrome(service=service, options=chrome_options)
  driver.set_page_load_timeout(60)

  return driver

## **States - Andhra Pradesh, Bihar, Gujarat**

In [4]:
states = ["ANDHRA PRADESH", "BIHAR", "GUJARAT"]

In [5]:
cols_names = {
    0: "Sno.",
    1: "Candidate",
    2: "Party",
    3: "Criminal Cases",
    4: "Education",
    5: "Age",
    6: "Total Assets",
    7: "Liabilities"
}
ua = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"}

## **2019 Data**

In [6]:
url_19 = "https://www.myneta.info/LokSabha2019/"
html_19 = requests.get(url_19, headers=ua, timeout=30)
print(html_19.status_code)

200


In [7]:
soup_19 = bs(html_19.text, "html.parser")

In [8]:
for card in soup_19.find_all("div", class_="w3-dropdown-click w3-block"):
    print(card.find("button").get_text(strip=True))
    break

ANDAMAN & NICOBAR ISLANDS


In [9]:
url_dict_19 = {}

In [10]:
found = 0
for card in soup_19.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_19[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_19)

{'ANDHRA PRADESH': 'index.php?action=show_constituencies&state_id=34', 'BIHAR': 'index.php?action=show_constituencies&state_id=37', 'GUJARAT': 'index.php?action=show_constituencies&state_id=39'}


In [11]:
data_19 = {}
for state in states:
    data_19[state] = []

In [12]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_19+url_dict_19[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_19[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the ne

In [13]:
print(len(data_19["ANDHRA PRADESH"]))
print(len(data_19["BIHAR"]))
print(len(data_19["GUJARAT"]))

314
619
371


In [14]:
andhra_19 = pd.DataFrame(data_19["ANDHRA PRADESH"])
bihar_19 = pd.DataFrame(data_19["BIHAR"])
gujarat_19 = pd.DataFrame(data_19["GUJARAT"])

In [15]:
andhra_19 = andhra_19.rename(columns=cols_names)

In [16]:
bihar_19 = bihar_19.rename(columns=cols_names)

In [17]:
gujarat_19 = gujarat_19.rename(columns=cols_names)

In [18]:
andhra_19

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,A. Hemanth,IND,0,5th Pass,28,"Rs 50,000\n~ 50 Thou+",Rs 0\n~
1,2,Abdul Waris,SDPI,4,8th Pass,58,"Rs 51,10,000\n~ 51 Lacs+",Rs 0\n~
2,3,Adala Prabhakara Reddy,YSRCP,0,12th Pass,70,"Rs 2,21,16,63,364\n~ 221 Crore+","Rs 15,22,81,255\n~ 15 Crore+"
3,4,Adari Anand Kumar,TDP,0,10th Pass,43,"Rs 41,05,47,212\n~ 41 Crore+","Rs 20,55,91,400\n~ 20 Crore+"
4,5,Adinarayana Inty,BJP,1,Post Graduate,67,"Rs 3,38,85,898\n~ 3 Crore+","Rs 44,79,155\n~ 44 Lacs+"
...,...,...,...,...,...,...,...,...
309,310,Yanamadala Venkata Suresh,IND,0,Graduate,50,"Rs 13,02,000\n~ 13 Lacs+",Rs 0\n~
310,311,Yarlagadda Rama Mohana Rao,Bahujan Maha Party,0,12th Pass,56,"Rs 1,00,86,000\n~ 1 Crore+",Rs 0\n~
311,312,Yeduguri Sandinti Avinash Reddy,YSRCP,4,Post Graduate,34,"Rs 18,69,69,794\n~ 18 Crore+","Rs 2,28,33,319\n~ 2 Crore+"
312,313,Yella Rao Siyyadula,IND,0,12th Pass,37,"Rs 57,000\n~ 57 Thou+",Rs 0\n~


In [ ]:
bihar_19

In [ ]:
gujarat_19

In [ ]:
# andhra_19.to_csv("raw/myneta_andhra_2019_17-09-2026.csv", index=False)
# bihar_19.to_csv("raw/myneta_bihar_2019_17-09-2026.csv", index=False)
# gujarat_19.to_csv("raw/myneta_gujarat_2019_17-09-2026.csv", index=False)

## **2014 Data**

In [ ]:
url_14 = "https://www.myneta.info/ls2014/"
html_14 = requests.get(url_14, headers=ua, timeout=30)
print(html_14.status_code)

In [ ]:
soup_14 = bs(html_14.text, "html.parser")

In [ ]:
url_dict_14 = {}
found = 0
for card in soup_14.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_14[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_14)

In [ ]:
data_14 = {}
for state in states:
    data_14[state] = []

In [ ]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_14+url_dict_14[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_14[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

In [ ]:
print(len(data_14["ANDHRA PRADESH"]))
print(len(data_14["BIHAR"]))
print(len(data_14["GUJARAT"]))

In [ ]:
andhra_14 = pd.DataFrame(data_14["ANDHRA PRADESH"])
bihar_14 = pd.DataFrame(data_14["BIHAR"])
gujarat_14 = pd.DataFrame(data_14["GUJARAT"])

In [ ]:
andhra_14 = andhra_14.rename(columns=cols_names)
bihar_14 = bihar_14.rename(columns=cols_names)
gujarat_14 = gujarat_14.rename(columns=cols_names)

In [ ]:
andhra_14

In [ ]:
bihar_14

In [ ]:
gujarat_14

In [ ]:
# andhra_14.to_csv("raw/myneta_andhra_2014_17-09-2026.csv", index=False)
# bihar_14.to_csv("raw/myneta_bihar_2014_17-09-2026.csv", index=False)
# gujarat_14.to_csv("raw/myneta_gujarat_2014_17-09-2026.csv", index=False)

## **2024 Data**

In [6]:
url_24 = "https://www.myneta.info/LokSabha2024/"
html_24 = requests.get(url_24, headers=ua, timeout=30)
print(html_24.status_code)

200


In [7]:
soup_24 = bs(html_24.text, "html.parser")

In [8]:
url_dict_24 = {}
found = 0
for card in soup_24.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_24[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_24)

{'ANDHRA PRADESH': 'index.php?action=show_constituencies&state_id=2', 'BIHAR': 'index.php?action=show_constituencies&state_id=5', 'GUJARAT': 'index.php?action=show_constituencies&state_id=11'}


In [9]:
data_24 = {}
for state in states:
    data_24[state] = []

In [10]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_24+url_dict_24[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_24[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the ne

In [11]:
print(len(data_24["ANDHRA PRADESH"]))
print(len(data_24["BIHAR"]))
print(len(data_24["GUJARAT"]))

450
496
266


In [12]:
andhra_24 = pd.DataFrame(data_24["ANDHRA PRADESH"])
bihar_24 = pd.DataFrame(data_24["BIHAR"])
gujarat_24 = pd.DataFrame(data_24["GUJARAT"])

In [13]:
andhra_24 = andhra_24.rename(columns=cols_names)
bihar_24 = bihar_24.rename(columns=cols_names)
gujarat_24 = gujarat_24.rename(columns=cols_names)

In [14]:
andhra_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,L. Khaja Hussain,IND,0,Graduate,42,"Rs 92,46,142\n~ 92 Lacs+",Rs 0\n~
1,2,A Chinna Moulali,BSP,0,Graduate Professional,43,"Rs 85,08,000\n~ 85 Lacs+","Rs 13,00,000\n~ 13 Lacs+"
2,3,A. Madhu,Jai Hindustan Party,0,10th Pass,31,"Rs 54,300\n~ 54 Thou+",Rs 0\n~
3,4,A. Purna Chandra,IND,0,10th Pass,37,"Rs 2,00,000\n~ 2 Lacs+",Rs 0\n~
4,5,A. Varaprasad,Jatiya Jana Sena Party,0,12th Pass,51,"Rs 2,53,427\n~ 2 Lacs+","Rs 1,23,000\n~ 1 Lacs+"
...,...,...,...,...,...,...,...,...
445,446,Yeddu Lakshminarayana,IND,0,5th Pass,66,"Rs 8,30,000\n~ 8 Lacs+","Rs 2,30,000\n~ 2 Lacs+"
446,447,Yedida Vara Prasad Kumar,Republican Party of India (A),0,12th Pass,55,"Rs 1,00,000\n~ 1 Lacs+",Rs 0\n~
447,448,Yelampalle Govardhan Reddy,All India Forward Bloc,0,12th Pass,48,"Rs 31,11,906\n~ 31 Lacs+",Rs 0\n~
448,449,Yenugupalli Krishna,Jaibhim Rao Bharat Party,4,Graduate,45,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...


In [15]:
bihar_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,Abu Bakar Rahmani,Country Citizen Party,0,Post Graduate,35,"Rs 13,58,312\n~ 13 Lacs+",Rs 0\n~
1,2,A. K. Singh (Ashok),IND,0,Graduate Professional,69,"Rs 2,20,62,267\n~ 2 Crore+",Rs 0\n~
2,3,Aakash Kumar Singh,INC,0,12th Pass,33,"Rs 6,59,33,186\n~ 6 Crore+",Rs 0\n~
3,4,Aanand Misra,IND,0,12th Pass,41,"Rs 56,66,811\n~ 56 Lacs+","Rs 1,70,000\n~ 1 Lacs+"
4,5,Abdhesh Kumar Roy,CPI,4,12th Pass,72,"Rs 45,54,459\n~ 45 Lacs+",Rs 0\n~
...,...,...,...,...,...,...,...,...
491,492,Vishnu Singh,Bharat Jago Janta Party,0,10th Pass,50,"Rs 31,28,369\n~ 31 Lacs+",Rs 0\n~
492,493,Vishwanath Tudu,IND,0,12th Pass,31,"Rs 1,00,612\n~ 1 Lacs+",Rs 0\n~
493,494,Vivek Thakur,BJP,0,Post Graduate,54,"Rs 5,05,66,297\n~ 5 Crore+","Rs 1,04,44,564\n~ 1 Crore+"
494,495,Yog Narayan Singh,IND,0,Literate,49,"Rs 26,95,200\n~ 26 Lacs+",Rs 0\n~


In [16]:
gujarat_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,Aarab Hasam Sumra,IND,0,5th Pass,62,"Rs 2,49,591\n~ 2 Lacs+",Rs 0\n~
1,2,Abdulhaq Ismail Nedariya,IND,1,5th Pass,46,"Rs 46,25,500\n~ 46 Lacs+",Rs 0\n~
2,3,Abdulkuddus,IND,0,5th Pass,49,"Rs 3,10,000\n~ 3 Lacs+",Rs 0\n~
3,4,Advocate J. P. Maraviya,INC,0,Graduate Professional,43,"Rs 1,02,31,739\n~ 1 Crore+","Rs 1,44,000\n~ 1 Lacs+"
4,5,Advocate Navinkumar Shankerbhai Patel,IND,0,Graduate Professional,53,"Rs 7,45,000\n~ 7 Lacs+",Rs 0\n~
...,...,...,...,...,...,...,...,...
261,262,Yusuf Vali Hasanali,IND,0,8th Pass,49,"Rs 27,31,000\n~ 27 Lacs+","Rs 6,00,000\n~ 6 Lacs+"
262,263,Zala Devrajbhai Babubhai,IND,3,8th Pass,34,"Rs 4,83,000\n~ 4 Lacs+",Rs 0\n~
263,264,Zala Nayan J.,IND,0,8th Pass,39,"Rs 1,61,640\n~ 1 Lacs+",Rs 0\n~
264,265,Zala Sanjaykumar Mahobatsinh,IND,0,Graduate,36,"Rs 4,00,000\n~ 4 Lacs+",Rs 0\n~


In [ ]:
# andhra_24.to_csv("raw/myneta_andhra_2024_20-09-2026.csv", index=False)
# bihar_24.to_csv("raw/myneta_bihar_2024_20-09-2026.csv", index=False)
# gujarat_24.to_csv("raw/myneta_gujarat_2024_20-09-2026.csv", index=False)

# Lok Sabha Winners Scraper — 2014, 2019, and 2024

Scrape the constituency-wise Lok Sabha winners for **Andhra Pradesh, Bihar, and Gujarat** across **2014, 2019, and 2024**.
The scraper uses **manual table selection**: first inspect the tables on each Wikipedia page, then enter the table index and exact column headings for each year/state. The final dataset contains only:

- State
- Election Year
- Constituency No
- Constituency
- Winner Candidate

**Special case:** The 2014 Andhra Pradesh page can include Telangana rows at the start. A configurable setting lets you skip the first 18 rows for that page only. Do not apply that skip to 2019 or 2024 unless inspection shows it is needed.

In [1]:
from io import StringIO
import time
import requests
from lxml import html
import pandas as pd
from IPython.display import display

YEARS = [2014, 2019, 2024]
STATES = ["Andhra Pradesh", "Bihar", "Gujarat"]

# Wikipedia article URLs for each year and state.
URLS = {
    (2014, "Andhra Pradesh"): "https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Andhra_Pradesh",
    (2014, "Bihar"): "https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Bihar",
    (2014, "Gujarat"): "https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Gujarat",
    (2019, "Andhra Pradesh"): "https://en.wikipedia.org/wiki/2019_Indian_general_election_in_Andhra_Pradesh",
    (2019, "Bihar"): "https://en.wikipedia.org/wiki/2019_Indian_general_election_in_Bihar",
    (2019, "Gujarat"): "https://en.wikipedia.org/wiki/2019_Indian_general_election_in_Gujarat",
    (2024, "Andhra Pradesh"): "https://en.wikipedia.org/wiki/2024_Indian_general_election_in_Andhra_Pradesh",
    (2024, "Bihar"): "https://en.wikipedia.org/wiki/2024_Indian_general_election_in_Bihar",
    (2024, "Gujarat"): "https://en.wikipedia.org/wiki/2024_Indian_general_election_in_Gujarat",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; EducationalElectionResultsScraper/1.0)"
}

def fetch_page(url):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.text

def flatten_columns(columns):
    """Convert regular or multi-level column headers into readable strings."""
    result = []
    for col in columns:
        parts = col if isinstance(col, tuple) else (col,)
        parts = [
            str(part).strip()
            for part in parts
            if str(part).strip()
            and not str(part).strip().startswith("Unnamed:")
        ]
        result.append(" | ".join(parts) if parts else "Unnamed")
    return result

print(f"Configured {len(URLS)} pages for {len(YEARS)} election years and {len(STATES)} states.")

Configured 9 pages for 3 election years and 3 states.


## Downloading pages and inspecting for table indices

In [2]:
PAGE_HTML = {}
PAGE_TABLES = {}
PAGE_ERRORS = {}

for (year, state), url in URLS.items():
    key = (year, state)
    print(f"\n{'=' * 100}\n{year} — {state}\nURL: {url}\n{'=' * 100}")

    try:
        html = fetch_page(url)
        PAGE_HTML[key] = html
        tables = pd.read_html(StringIO(html))
        PAGE_TABLES[key] = tables

        # print(f"Found {len(tables)} tables.")
        for index, table in enumerate(tables):
            # print(f"\nTable index: {index} | shape: {table.shape}")
            print("Columns:", flatten_columns(table.columns))
            # display(table.head(2))
    except Exception as exc:
        PAGE_ERRORS[key] = str(exc)
        print(f"ERROR loading this page: {exc}")

    time.sleep(1)  # Be polite to Wikipedia

if PAGE_ERRORS:
    print("\nPages needing attention:")
    for key, error in PAGE_ERRORS.items():
        print(f"{key}: {error}")


2014 — Andhra Pradesh
URL: https://en.wikipedia.org/wiki/2014_Indian_general_election_in_Andhra_Pradesh
Columns: ['0', '1', '2', '3']
Columns: ['Unnamed', 'First party', 'Second party', 'Third party']
Columns: ['0', '1']
Columns: ['Alliance/Party', 'Alliance/Party.1', 'Alliance/Party.2', 'Alliance/Party.3', 'Flag', 'Symbol', 'Leader', 'Seats contested', 'Seats contested.1', 'Unnamed']
Columns: ['Constituency | #', 'Constituency | Name', 'TRS | Party', 'TRS | Party.1', 'TRS | Candidate', 'UPA | Party', 'UPA | Party.1', 'UPA | Candidate', 'NDA | Party', 'NDA | Party.1', 'NDA | Candidate', 'YSRCP | Party', 'YSRCP | Party.1', 'YSRCP | Candidate']
Columns: ['Constituency | #', 'Constituency | Name', 'YSRCP | Party', 'YSRCP | Party.1', 'YSRCP | Candidate', 'NDA | Party', 'NDA | Party.1', 'NDA | Candidate', 'UPA | Party', 'UPA | Party.1', 'UPA | Candidate']
Columns: ['Conducted in Month(s) | Conducted in Month(s)', 'Ref | Ref', 'Polling Organisation/Agency | Polling Organisation/Agency', 'Sa

## Manual configuration — table indexes, column headings, and row skips

- TABLE_INDEX[(year, state)]: numeric table index for that page.
- COLUMN_MAP[(year, state)]: maps exact heading from above → desired output field.
- SKIP_ROWS[(year, state)]: number of leading data rows to skip. The default below skips 18 rows only for **2014 Andhra Pradesh**, to account for split of Andhra and Telangana.


In [3]:
# 1) Replace None with the correct table index for every page after inspecting Step 3.
TABLE_INDEX = {
    (2014, "Andhra Pradesh"): 9,
    (2014, "Bihar"): 9,
    (2014, "Gujarat"): 7,
    (2019, "Andhra Pradesh"): 6,
    (2019, "Bihar"): 10,
    (2019, "Gujarat"): 7,
    (2024, "Andhra Pradesh"): 13,
    (2024, "Bihar"): 14,
    (2024, "Gujarat"): 14,
    
}

# 2) Exact source heading -> final output column.
# Update each mapping to match the headings printed in Step 3.
DEFAULT_COLUMN_MAP = {
    "Constituency | #": "Constituency No.",
    "Constituency | Name": "Constituency",
    "Winner | Candidate": "Winner Candidate",
        "Winner | Party.1": "Winner Party",
}

COLUMN_MAP = {
    (2014, "Andhra Pradesh"): DEFAULT_COLUMN_MAP.copy(),
    (2014, "Bihar"): DEFAULT_COLUMN_MAP.copy(),
    (2014, "Gujarat"): DEFAULT_COLUMN_MAP.copy(),
    (2019, "Andhra Pradesh"): {
        "Constituency | Constituency": "Constituency No.",
        "Constituency | Constituency.1": "Constituency",
        "Winner | Candidate": "Winner Candidate",
        "Winner | Party.1": "Winner Party",
    },
    (2019, "Bihar"): DEFAULT_COLUMN_MAP.copy(),
    (2019, "Gujarat"): {
        "Constituency" : "Constituency",
        "No": "Constituency No.",
        "Winner": "Winner Candidate",
        "Party.1": "Winner Party",
    },
    (2024, "Andhra Pradesh"): DEFAULT_COLUMN_MAP.copy(),
    (2024, "Bihar"): DEFAULT_COLUMN_MAP.copy(),
    (2024, "Gujarat"): DEFAULT_COLUMN_MAP.copy(),
}

# 3) Number of leading data rows to skip per page.
# This preserves your earlier Andhra Pradesh 2014 workaround.
SKIP_ROWS = {
    (2014, "Andhra Pradesh"): 17,
    (2014, "Bihar"): 0,
    (2014, "Gujarat"): 0,
    (2019, "Andhra Pradesh"): 0,
    (2019, "Bihar"): 0,
    (2019, "Gujarat"): 0,
}

STRICT_COLUMN_CHECK = True

## Check for valid table and table headings

In [4]:
SELECTED_TABLES = {}

for key, url in URLS.items():
    year, state = key

    if key not in PAGE_TABLES:
        print(f"Skipping {year} {state}: page did not load successfully.")
        continue

    index = TABLE_INDEX.get(key)
    if index is None:
        raise ValueError(
            f"Set TABLE_INDEX[{key!r}] in Step 4 after inspecting the page tables."
        )

    tables = PAGE_TABLES[key]
    if not isinstance(index, int) or not (0 <= index < len(tables)):
        raise IndexError(
            f"{year} {state}: table index {index!r} is invalid. "
            f"Choose an index from 0 to {len(tables) - 1}."
        )

    table = tables[index].copy()
    table.columns = flatten_columns(table.columns)

    skip = SKIP_ROWS.get(key, 0)
    if skip:
        table = table.iloc[skip:].reset_index(drop=True)

    mapping = COLUMN_MAP.get(key, {})
    available = list(table.columns)
    missing = [heading for heading in mapping if heading not in available]

    print(f"\n{year} — {state}: table {index}, shape after row skip={table.shape}")
    print("Selected headings:", available)

    if missing:
        message = f"{year} {state}: headings not found: {missing}"
        if STRICT_COLUMN_CHECK:
            raise ValueError(message + "\nUpdate COLUMN_MAP with exact headings from Step 3.")
        print("WARNING:", message)

    SELECTED_TABLES[key] = table
    display(table.head(3))


2014 — Andhra Pradesh: table 9, shape after row skip=(26, 15)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Poll% | Poll%', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Poll% | Poll%,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,18.0,Araku,72.92,Kothapalli Geetha,NaN,YCP,413191.0,45.42,Gummadi Sandhya Rani,NaN,TDP,259557.0,24.59,171290.0,16.38
1,19.0,Srikakulam,74.36,Ram Mohan Naidu Kinjarapu,NaN,TDP,556163.0,52.90,Reddy Shanthi,NaN,YCP,428591.0,40.76,127572.0,12.14
2,20.0,Vizianagaram,79.79,Ashok Gajapathi Raju,NaN,TDP,536549.0,47.89,Ravu Venkata Swetha,NaN,YCP,429638.0,38.35,106911.0,9.54



2014 — Bihar: table 9, shape after row skip=(41, 14)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1.0,Valmiki Nagar,Satish Chandra Dubey,NaN,BJP,364013.0,40.44,Purnmasi Ram,NaN,INC,246218.0,27.35,117795.0,13.09
1,2.0,Paschim Champaran,Sanjay Jaiswal,NaN,BJP,371232.0,43.43,Prakash Jha,NaN,JD(U),260978.0,30.53,110254.0,12.90
2,3.0,Purvi Champaran,Radha Mohan Singh,NaN,BJP,400452.0,48.68,Binod Kumar Srivastava,NaN,RJD,208289.0,25.32,192163.0,23.36



2014 — Gujarat: table 7, shape after row skip=(27, 14)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1.0,Kachchh,Vinodbhai Chavda,NaN,BJP,562855.0,59.40,Dinesh Parmar,NaN,INC,308373.0,32.55,254482.0,26.85
1,2.0,Banaskantha,Haribhai Chaudhary,NaN,BJP,507856.0,57.23,Joitabhai Patel,NaN,INC,305522.0,34.43,202334.0,22.80
2,3.0,Patan,Liladhar Vaghela,NaN,BJP,518538.0,54.21,Bhavsinh Rathod,NaN,INC,379819.0,39.70,138719.0,14.51


... (8 outputs left out)


2024 — Bihar: table 14, shape after row skip=(40, 15)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Constituency | Poll\xa0%', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Constituency | Poll %,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1,Valmiki Nagar,60.33,Sunil Kumar,NaN,JD(U),523422,47.50,Deepak Vadav,NaN,RJD,424747,38.55,98675,8.95
1,2,Paschim Champaran,61.78,Sanjay Jaiswal,NaN,BJP,580421,53.43,Madan Mohan Tiwari,NaN,INC,443853,40.86,136568,12.57
2,3,Purvi Champaran,59.90,Radha Mohan Singh,NaN,BJP,580421,50.50,Rajesh Kumar,NaN,VIP,443853,42.28,88287,8.22



2024 — Gujarat: table 14, shape after row skip=(27, 14)
Selected headings: ['Constituency | #', 'Constituency | Name', 'Winner | Candidate', 'Winner | Party', 'Winner | Party.1', 'Winner | Votes', 'Winner | %', 'Runner Up | Candidate', 'Runner Up | Party', 'Runner Up | Party.1', 'Runner Up | Votes', 'Runner Up | %', 'Margin | Margin', '% | %']


,Constituency | #,Constituency | Name,Winner | Candidate,Winner | Party,Winner | Party.1,Winner | Votes,Winner | %,Runner Up | Candidate,Runner Up | Party,Runner Up | Party.1,Runner Up | Votes,Runner Up | %,Margin | Margin,% | %
0,1.0,Kachchh,Vinodbhai Chavda,NaN,BJP,659574,60.18,Nitishbhai Lalan,NaN,INC,390792,35.66,268782,24.52
1,2.0,Banaskantha,Geni Thakor,NaN,INC,671883,48.75,Rekhaben Chaudhary,NaN,BJP,641477,46.55,30406,2.20
2,3.0,Patan,Bharatsinhji Dabhi,NaN,BJP,591947,49.51,Chandanji Thakor,NaN,INC,560071,46.85,31876,2.66


## Extract the five required columns and combine all years

Party names are normalised, some tables contain abbreviations while some contain complete name.

In [5]:
PARTY_ALIASES = {"Bharatiya Janata Party": "BJP", "HAM(S)": "Hindustani Awam Morcha (Secular)", "JSP": "Janasena Party", "LJP(RV)": "Lok Janshakti Party(Ram Vilas)", "RLSP": "Rashtriya Lok Samta Party", "YCP": "YSRCP"}

def extract_results(key, table):
    year, state = key
    mapping = COLUMN_MAP[key]

    required_outputs = {
        "Constituency No.",
        "Constituency",
        "Winner Candidate",
        "Winner Party",
    }
    missing_outputs = required_outputs - set(mapping.values())
    if missing_outputs:
        raise ValueError(
            f"{year} {state}: COLUMN_MAP must produce these fields: "
            f"{sorted(required_outputs)}. Missing: {sorted(missing_outputs)}"
        )

    missing_inputs = [heading for heading in mapping if heading not in table.columns]
    if missing_inputs:
        raise KeyError(
            f"{year} {state}: missing source headings {missing_inputs}. "
            f"Available: {list(table.columns)}"
        )

    # Select and rename the configured source columns.
    result = table[list(mapping.keys())].rename(columns=mapping)

    # Clean whitespace in text values.
    for col in result.columns:
        result[col] = result[col].map(
            lambda value: value.strip() if isinstance(value, str) else value
        )

    result["Winner Party Source"] = result["Winner Party"]
    result["Winner Party"] = result["Winner Party"].replace(PARTY_ALIASES)

    result = result.dropna(how="all").copy()
    result.insert(0, "Election Year", year)
    result.insert(0, "State", state)

    return result[
        [
            "State",
            "Election Year",
            "Constituency No.",
            "Constituency",
            "Winner Candidate",
            "Winner Party",
            "Winner Party Source",
        ]
    ].reset_index(drop=True)


results_by_page = {
    key: extract_results(key, table)
    for key, table in SELECTED_TABLES.items()
}

combined_df = pd.concat(results_by_page.values(), ignore_index=True)

print("Combined dataset shape:", combined_df.shape)
display(combined_df.head(15))
print("\nRows by year and state:")
display(combined_df.groupby(["Election Year", "State"]).size().rename("Rows").to_frame())

Combined dataset shape: (273, 7)


,State,Election Year,Constituency No.,Constituency,Winner Candidate,Winner Party,Winner Party Source
0,Andhra Pradesh,2014,18.0,Araku,Kothapalli Geetha,YSRCP,YCP
1,Andhra Pradesh,2014,19.0,Srikakulam,Ram Mohan Naidu Kinjarapu,TDP,TDP
2,Andhra Pradesh,2014,20.0,Vizianagaram,Ashok Gajapathi Raju,TDP,TDP
3,Andhra Pradesh,2014,21.0,Visakhapatnam,Kambhampati Hari Babu,BJP,BJP
4,Andhra Pradesh,2014,22.0,Anakapalle,Muttamsetti Srinivasa Rao,TDP,TDP
5,Andhra Pradesh,2014,23.0,Kakinada,Thota Narasimham,TDP,TDP
6,Andhra Pradesh,2014,24.0,Amalapuram,Pandula Ravindra Babu,TDP,TDP
7,Andhra Pradesh,2014,25.0,Rajahmundry,Murali Mohan,TDP,TDP
8,Andhra Pradesh,2014,26.0,Narasapuram,Gokaraju Ganga Raju,BJP,BJP
9,Andhra Pradesh,2014,27.0,Eluru,Maganti Venkateswara Rao,TDP,TDP



Rows by year and state:


Rows
Election Year State               
2014          Andhra Pradesh    25
              Bihar             40
              Gujarat           26
2019          Andhra Pradesh    25
              Bihar             40
              Gujarat           26
2024          Andhra Pradesh    25
              Bihar             40
              Gujarat           26

## Export to CSV

In [ ]:
CSV_PATH = "raw/wikipedia_myneta_loksabha_winners_2014_2019_2024_normalized_20-09-2026.csv"

# combined_df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print("Saved:", CSV_PATH)

Saved: loksabha_winners_2014_2019_2024_normalized.csv
